# Sparse-Aware Bi-LSTM for ICESat-2 Sea Ice Classification

**Based on:** `lstm_prof_style.ipynb` + all fine-tuning improvements from
`final.ipynb`.

**The problem this notebook solves:** ICESat-2 data has natural gaps —
cloud cover, dead detector channels, and low-signal regions leave some
10 m segments with zero photon returns (`pcnt_mean == 0`).  A standard
LSTM processes these dead steps identically to real data, corrupting the
hidden state with noise.

**Design:**

```
                     ┌─ valid_mask (B, T) ─────────────────────┐
                     │                                          │
  raw window         │  zero-fill         Bi-LSTM              │  masked
  (B, T, 8)  ──mul──►│  invalid steps ──► hidden (B, T, 128) ──► mean pool ──► feat (B, 128)
                     │                                          │
                     └──────────────────────────────────────────┘
                                                                 + valid_ratio (B, 1)
                                                                        │
                                                            Linear(129, 32)  ELU  Dropout
                                                            Linear(32,  32)  ELU  Dropout
                                                            Linear(32,   3)   ← logits
```

**Validity rule:** a segment is valid when `pcnt_mean > 0`
AND all 8 feature columns are finite (no NaN / inf).

**Masked mean pooling:** aggregate hidden states only over valid steps.
If a window contains *no* valid steps (fully clouded-out track), fall
back to the center-step hidden state so the forward pass never returns
NaN.

**valid_ratio feature:** the fraction of valid steps (0.0–1.0) is
appended to the pooled vector.  This lets the head learn to be less
confident when the photon track is mostly missing.


## 0. Setup

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")


In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print("torch:", torch.__version__,
      "cuda:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1. Config

In [ ]:
PROJECT_ROOT = Path("/home/spant/Research Seminar/Project")
EXP_NAME     = "lstm_sparse_aware_v1"

CSV_DIR  = PROJECT_ROOT / "IS2_Corrected_data"
RUN_DIR  = PROJECT_ROOT / "runs" / EXP_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ---- data ----
SEED          = 42
NUM_CLASSES   = 3
SEQ_LEN       = 9
NEARBY        = SEQ_LEN // 2
GROUPED_SPLIT = True

# ---- training ----
BATCH_SIZE    = 32
EPOCHS        = 80
LR            = 8.886176350890356e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 12
NUM_WORKERS   = 2

# ---- architecture ----
LSTM_HIDDEN   = 64      # bidirectional → 128-dim output
LSTM_DROPOUT  = 0.4
HEAD_HIDDEN   = 32
HEAD_DROPOUT  = 0.4
GRAD_CLIP     = 1.0

# ---- focal loss ----
ALPHA = [0.05, 0.45, 0.60]
GAMMA = 2.0

CLASS_NAMES      = ["ice", "thin_ice", "water"]
CLASS_NAMES_DISP = ["thick ice", "thin ice", "water"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Run dir:", RUN_DIR)
print(f"SEQ_LEN={SEQ_LEN}  Bi-LSTM hidden={LSTM_HIDDEN}  clip={GRAD_CLIP}")
print(f"Focal alpha={ALPHA}  gamma={GAMMA}")


## 2. Load 10 m segment CSVs

Same 8 features as prof-style.  After engineering `h_diff` and
`rel_height_min_elev`, we also compute a boolean `valid` column per
segment:

```python
valid = (pcnt_mean > 0) & all_features_finite
```

This column is carried through the sliding-window builder so each window
knows which of its steps had real photon returns.


In [ ]:
FEATURES = [
    "h_cor_mean", "h_diff", "rel_height_min_elev", "height_sd",
    "pcnth_mean", "pcnt_mean", "bcnt_mean", "brate_mean",
]
N_FEATS = len(FEATURES)


def load_segment_csv(csv_path: Path):
    df = pd.read_csv(csv_path)
    needed = ["h_cor_mean", "h_cor_med", "x_atc", "label",
              "height_sd", "pcnth_mean", "pcnt_mean", "bcnt_mean", "brate_mean"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")
    df = df[df["label"].isin([0, 1, 2])].copy()
    df = df.dropna(subset=needed)
    df = df.sort_values("x_atc").reset_index(drop=True)

    # derived features (same as prof-style)
    df["h_diff"]              = df["h_cor_mean"] - df["h_cor_med"]
    df["rel_height_min_elev"] = df["h_cor_mean"] - df["h_cor_mean"].min()

    # validity flag: real photon return AND no NaN in feature columns
    has_signal  = df["pcnt_mean"] > 0
    all_finite  = df[FEATURES].apply(np.isfinite).all(axis=1)
    df["valid"] = (has_signal & all_finite).astype(np.float32)
    return df


csv_files = sorted(CSV_DIR.glob("ATL03_*_done.csv"))
print(f"found {len(csv_files)} CSVs")

records = []
for p in csv_files:
    parts = p.stem.split("_")
    tile  = parts[3]; beam = parts[4]
    seg   = load_segment_csv(p)
    seg["tile"] = tile; seg["beam"] = beam; seg["src"] = p.name
    n_valid = int(seg["valid"].sum())
    pct_valid = 100.0 * n_valid / max(len(seg), 1)
    records.append((tile, beam, seg))
    print(f"  {p.name}: {len(seg):,} segs  valid={n_valid} ({pct_valid:.1f}%)"
          f"  labels={seg['label'].value_counts().to_dict()}")
print(f"total: {sum(len(s) for _, _, s in records):,} segments")


## 3. Build (N, SEQ_LEN, 8) windows + valid masks

Each sample is a tuple `(X_window, valid_mask)`:
- `X_window`: (SEQ_LEN, 8) feature matrix — **invalid steps are zeroed**
- `valid_mask`: (SEQ_LEN,) float32 — 1.0 for valid steps, 0.0 for gaps

Zeroing invalid steps ensures the LSTM sees zeros (not whatever garbage
was in the CSV) for clouded-out segments, and the mask tells the pooling
layer to ignore those hidden states.


In [ ]:
def sliding_windows(seg_df, seq_len=SEQ_LEN, nearby=NEARBY):
    arr   = seg_df[FEATURES].to_numpy(dtype=np.float32)  # (T, 8)
    lab   = seg_df["label"].to_numpy(dtype=np.int64)      # (T,)
    valid = seg_df["valid"].to_numpy(dtype=np.float32)    # (T,)

    if len(arr) < seq_len:
        return (np.empty((0, seq_len, N_FEATS), dtype=np.float32),
                np.empty((0, seq_len),          dtype=np.float32),
                np.empty((0,),                  dtype=np.int64))

    n = len(arr) - 2 * nearby

    X    = np.zeros((n, seq_len, N_FEATS), dtype=np.float32)
    mask = np.zeros((n, seq_len),          dtype=np.float32)
    for k in range(seq_len):
        X[:, k, :]  = arr[k : k + n]
        mask[:, k]  = valid[k : k + n]

    # zero-fill invalid steps in the feature matrix
    X = X * mask[:, :, None]

    y = lab[nearby : nearby + n]
    keep = (y >= 0) & (y < NUM_CLASSES)
    return X[keep], mask[keep], y[keep]


bundles = []
for tile, beam, seg in records:
    X, mask, y = sliding_windows(seg)
    if len(X) == 0:
        continue
    bundles.append({"tile": tile, "beam": beam, "X": X, "mask": mask, "y": y})

total_n = sum(len(b["y"]) for b in bundles)
# sparsity stats across all windows
all_masks = np.concatenate([b["mask"] for b in bundles])
print(f"{len(bundles)} tracks, {total_n:,} samples")
print(f"mean valid steps per window: {all_masks.sum(1).mean():.2f} / {SEQ_LEN}")
print(f"fully-sparse windows (0 valid): {(all_masks.sum(1)==0).sum():,} "
      f"({100*(all_masks.sum(1)==0).mean():.1f}%)")


## 4. Train / val / test split (tile-grouped)

In [ ]:
train_tiles = {"T02CNA", "T02CNC"}
test_tiles  = {"T03CWT"}

X_tr    = np.concatenate([b["X"]    for b in bundles if b["tile"] in train_tiles])
mask_tr = np.concatenate([b["mask"] for b in bundles if b["tile"] in train_tiles])
y_tr    = np.concatenate([b["y"]    for b in bundles if b["tile"] in train_tiles])

X_te    = np.concatenate([b["X"]    for b in bundles if b["tile"] in test_tiles])
mask_te = np.concatenate([b["mask"] for b in bundles if b["tile"] in test_tiles])
y_te    = np.concatenate([b["y"]    for b in bundles if b["tile"] in test_tiles])

rng = np.random.RandomState(SEED)
idx = rng.permutation(len(X_tr))
cut = int(0.10 * len(idx))
vi, ti = idx[:cut], idx[cut:]
X_val, mask_val, y_val = X_tr[vi], mask_tr[vi], y_tr[vi]
X_tr,  mask_tr,  y_tr  = X_tr[ti], mask_tr[ti], y_tr[ti]

print(f"train: {len(X_tr):,}   val: {len(X_val):,}   test: {len(X_te):,}")
for name, y in [("train", y_tr), ("val", y_val), ("test", y_te)]:
    c = np.bincount(y, minlength=NUM_CLASSES)
    p = c / max(c.sum(), 1) * 100
    print(f"  {name}: " + "  ".join(f"{n}={v} ({pv:.1f}%)"
          for n, v, pv in zip(CLASS_NAMES, c, p)))


## 5. Standardize features (z-score, train set only, valid steps only)

In [ ]:
# compute stats only from valid (non-zero) entries to avoid bias from the
# zero-fill padding introduced for invalid steps
valid_rows = X_tr[mask_tr.astype(bool).any(axis=1)].reshape(-1, N_FEATS)
# filter to rows that came from valid steps
flat_valid = X_tr.reshape(-1, N_FEATS)
flat_mask  = mask_tr.reshape(-1)
valid_only = flat_valid[flat_mask > 0]
means = valid_only.mean(axis=0) if len(valid_only) else flat_valid.mean(axis=0)
stds  = valid_only.std(axis=0)  + 1e-6


def standardize(X, mask):
    # only standardize steps that are valid; leave zeroed steps as zero
    X_s = (X - means[None, None, :]) / stds[None, None, :]
    return X_s * mask[:, :, None]    # re-zero invalid steps after standardizing


X_tr   = standardize(X_tr,  mask_tr).astype(np.float32)
X_val  = standardize(X_val, mask_val).astype(np.float32)
X_te   = standardize(X_te,  mask_te).astype(np.float32)
print("means:", means.round(3))
print("stds: ", stds.round(3))


## 6. Dataset + DataLoaders

In [ ]:
class SparseSegmentDataset(Dataset):
    def __init__(self, X, mask, y):
        self.X    = torch.from_numpy(X)
        self.mask = torch.from_numpy(mask)
        self.y    = torch.from_numpy(y).long()

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.mask[i], self.y[i]


train_loader = DataLoader(SparseSegmentDataset(X_tr,  mask_tr,  y_tr),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(SparseSegmentDataset(X_val, mask_val, y_val),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
test_loader  = DataLoader(SparseSegmentDataset(X_te,  mask_te,  y_te),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
print(f"train steps: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")


## 7. Model — SparseLSTM

**Key difference from standard LSTM notebooks:** the `forward` method
accepts a `valid_mask` tensor and uses it to:

1. **Zero-fill** invalid steps before the LSTM (already done in data prep,
   but applied again here so the model is safe when called standalone).
2. **Masked mean pooling** — average only over hidden states at valid steps.
3. **valid_ratio** — fraction of valid steps concatenated to the pooled
   vector so the head can modulate confidence.

Fallback: if a window has zero valid steps the model uses the center hidden
state (deterministic, avoids NaN).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SparseLSTM(nn.Module):
    # Bi-LSTM with masked mean pooling over valid ICESat-2 steps.

    def __init__(self, n_features=N_FEATS, hidden=LSTM_HIDDEN,
                 head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
                 lstm_dropout=LSTM_DROPOUT, num_classes=NUM_CLASSES):
        super().__init__()
        self.hidden = hidden
        self.center = SEQ_LEN // 2

        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=1, batch_first=True,
                            bidirectional=True)
        out_dim = hidden * 2 + 1  # +1 for valid_ratio

        self.dropout = nn.Dropout(lstm_dropout)
        self.head = nn.Sequential(
            nn.Linear(out_dim, head_hidden), nn.ELU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, head_hidden), nn.ELU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, num_classes),
        )

    def forward(self, x, valid_mask=None):
        # x: (B, T, F)   valid_mask: (B, T) float32  1=valid 0=gap
        if valid_mask is not None:
            # ensure invalid steps are zeroed (safety re-apply)
            x = x * valid_mask.unsqueeze(-1)

        out, _ = self.lstm(x)           # (B, T, hidden*2)
        out    = self.dropout(out)

        if valid_mask is not None:
            # masked mean pooling over valid steps
            m       = valid_mask.unsqueeze(-1)          # (B, T, 1)
            count   = m.sum(dim=1).clamp(min=1.0)       # (B, 1)
            pooled  = (out * m).sum(dim=1) / count       # (B, hidden*2)

            # fallback: if a window has no valid steps, use center step
            no_valid = (valid_mask.sum(dim=1) == 0)     # (B,)
            if no_valid.any():
                center_feat = out[no_valid, self.center, :]
                pooled[no_valid] = center_feat

            # valid_ratio feature: (B, 1)
            valid_ratio = valid_mask.mean(dim=1, keepdim=True)
        else:
            pooled      = out.mean(dim=1)
            valid_ratio = torch.ones(x.size(0), 1, device=x.device)

        feat = torch.cat([pooled, valid_ratio], dim=1)  # (B, hidden*2 + 1)
        return self.head(feat)


model = SparseLSTM().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"SparseLSTM parameters: {n_params:,}")
print(model)


## 8. Focal Loss + Adam + ReduceLROnPlateau

In [ ]:
class CategoricalFocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma = float(gamma)

    def forward(self, logits, target):
        log_probs  = F.log_softmax(logits, dim=1)
        log_p_t    = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t        = log_p_t.exp().clamp(min=1e-8, max=1.0 - 1e-8)
        alpha_t    = self.alpha.to(logits.device)[target]
        per_sample = -alpha_t * (1.0 - p_t).pow(self.gamma) * log_p_t
        return per_sample.mean()


criterion = CategoricalFocalLoss(alpha=ALPHA, gamma=GAMMA).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6, verbose=True)
print("loss + optimizer + scheduler ready")


## 9. Training loop

In [ ]:
def cm_accum(cm, logits, targets):
    pred = logits.argmax(1).detach().cpu().numpy().ravel()
    t    = targets.detach().cpu().numpy().ravel()
    idx  = NUM_CLASSES * t + pred
    cm  += np.bincount(idx, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_cm(cm):
    iou = []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
        denom = tp + fp + fn
        iou.append(float(tp / denom) if denom > 0 else 0.0)
    iou = np.array(iou, dtype=np.float64)
    pix_acc = float(np.diag(cm).sum() / max(cm.sum(), 1))
    return float(iou.mean()), iou, pix_acc


best_val      = -1.0
patience_left = PATIENCE
log           = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    # ---- train ----
    model.train()
    tr_loss = 0.0; n_seen = 0
    tr_cm   = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for X, mask, y in train_loader:
        X    = X.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        y    = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(X, mask)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        tr_loss += loss.item() * X.size(0); n_seen += X.size(0)
        cm_accum(tr_cm, logits, y)
    tr_loss /= max(n_seen, 1)
    tr_miou, _, tr_acc = metrics_from_cm(tr_cm)

    # ---- val ----
    model.eval()
    va_loss = 0.0; n_seen = 0
    va_cm   = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for X, mask, y in val_loader:
            X    = X.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            y    = y.to(device, non_blocking=True)
            logits   = model(X, mask)
            va_loss += criterion(logits, y).item() * X.size(0)
            n_seen  += X.size(0)
            cm_accum(va_cm, logits, y)
    va_loss /= max(n_seen, 1)
    va_miou, va_iou, va_acc = metrics_from_cm(va_cm)

    scheduler.step(va_miou)
    current_lr = optimizer.param_groups[0]["lr"]

    log.append({"epoch": epoch,
                "train_loss": tr_loss, "train_miou": tr_miou, "train_acc": tr_acc,
                "val_loss":   va_loss, "val_miou":   va_miou, "val_acc":   va_acc,
                "val_ice": va_iou[0], "val_thin": va_iou[1], "val_water": va_iou[2],
                "lr": current_lr})
    print(f"ep {epoch:03d}  tr_loss {tr_loss:.4f}  tr_mIoU {tr_miou:.4f}  |  "
          f"va_loss {va_loss:.4f}  va_mIoU {va_miou:.4f}  "
          f"[{va_iou.round(3).tolist()}]  lr={current_lr:.2e}  "
          f"({time.perf_counter()-t0:.0f}s)")

    if va_miou > best_val + 1e-4:
        best_val      = va_miou
        patience_left = PATIENCE
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "alpha": ALPHA, "gamma": GAMMA,
                    "means": means, "stds": stds,
                    "val_metrics": {"miou": va_miou, "per_iou": va_iou.tolist(),
                                    "pix_acc": va_acc, "loss": va_loss}},
                   RUN_DIR / "best.pt")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stop at epoch {epoch}. Best val mIoU = {best_val:.4f}")
            break

pd.DataFrame(log).to_csv(RUN_DIR / "metrics.csv", index=False)
print(f"\nBest val mIoU: {best_val:.4f}  saved → best.pt")


## 10. Test evaluation (best.pt)

In [ ]:
ck = torch.load(RUN_DIR / "best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["model_state"]); model.eval()
print(f"Loaded epoch {ck['epoch']}  (val mIoU {ck['val_metrics']['miou']:.4f})")

test_cm   = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
test_loss = 0.0; n_seen = 0
with torch.no_grad():
    for X, mask, y in test_loader:
        X    = X.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        y    = y.to(device, non_blocking=True)
        logits     = model(X, mask)
        test_loss += criterion(logits, y).item() * X.size(0)
        n_seen    += X.size(0)
        cm_accum(test_cm, logits, y)
test_loss /= max(n_seen, 1)
test_miou, test_iou, test_acc = metrics_from_cm(test_cm)

prec = np.zeros(NUM_CLASSES); rec = np.zeros(NUM_CLASSES); f1 = np.zeros(NUM_CLASSES)
for c in range(NUM_CLASSES):
    tp = test_cm[c, c]; fp = test_cm[:, c].sum() - tp; fn = test_cm[c, :].sum() - tp
    prec[c] = tp / (tp + fp) if (tp + fp) else 0.0
    rec[c]  = tp / (tp + fn) if (tp + fn) else 0.0
    f1[c]   = 2*prec[c]*rec[c] / (prec[c]+rec[c]) if (prec[c]+rec[c]) else 0.0

print(f"\nTEST  mIoU {test_miou:.4f}  pix_acc {test_acc:.4f}  loss {test_loss:.4f}")
print(f"per-class IoU       : {test_iou.round(4).tolist()}")
print(f"per-class precision : {prec.round(4).tolist()}")
print(f"per-class recall    : {rec.round(4).tolist()}")
print(f"per-class F1        : {f1.round(4).tolist()}")
print(f"macro-F1            : {f1.mean():.4f}")

with open(RUN_DIR / "test_metrics.json", "w") as f:
    json.dump({"miou": test_miou, "per_iou": test_iou.tolist(),
               "pix_acc": test_acc, "loss": test_loss,
               "precision": prec.tolist(), "recall": rec.tolist(),
               "f1": f1.tolist(), "macro_f1": float(f1.mean()),
               "alpha": ALPHA, "gamma": GAMMA,
               "grouped_split": GROUPED_SPLIT}, f, indent=2)


## 11. Confusion matrix (prof-style)

In [ ]:
def plot_cm_percent(cm, title, save_path):
    pct      = cm.astype(np.float64)
    row_sums = pct.sum(axis=1, keepdims=True)
    pct      = np.where(row_sums > 0, pct / np.maximum(row_sums, 1) * 100.0, 0.0)
    fig, ax  = plt.subplots(figsize=(7.2, 5.8))
    im       = ax.imshow(pct, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels([f"Predicted {c}" for c in CLASS_NAMES_DISP], fontsize=11)
    ax.set_yticklabels(CLASS_NAMES_DISP, fontsize=11)
    ax.set_xlabel("Predicted", fontsize=12); ax.set_ylabel("Actual", fontsize=12)
    ax.set_title(title, fontsize=13)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = pct[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v > 55 else "black", fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


plot_cm_percent(test_cm, "Confusion Matrix (Percentages) — Sparse-Aware LSTM",
                RUN_DIR / "confmat.png")


## 12. Loss + mIoU curves

In [ ]:
hist = pd.read_csv(RUN_DIR / "metrics.csv")
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(hist["epoch"], hist["train_loss"], label="train")
axes[0].plot(hist["epoch"], hist["val_loss"],   label="val")
axes[0].set_title("Focal loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist["epoch"], hist["train_miou"], label="train")
axes[1].plot(hist["epoch"], hist["val_miou"],   label="val")
axes[1].plot(hist["epoch"], hist["val_thin"],   "--", label="val thin_ice", alpha=0.7)
axes[1].set_title("mIoU"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(hist["epoch"], hist["lr"])
axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch")
axes[2].set_yscale("log"); axes[2].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig(RUN_DIR / "loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()


## 13. Sparsity analysis — performance by valid_ratio bucket

Split test predictions into buckets by what fraction of the window was
valid (0–20 %, 20–40 %, …, 80–100 %) and show mIoU per bucket.
This tells us whether the masked-pooling design actually helps on the
hardest (most-sparse) windows.


In [ ]:
# Re-run inference, collecting per-sample prediction + valid_ratio
preds_all  = []
labels_all = []
ratios_all = []

model.eval()
with torch.no_grad():
    for X, mask, y in test_loader:
        X    = X.to(device)
        mask = mask.to(device)
        logits = model(X, mask)
        preds_all.append(logits.argmax(1).cpu().numpy())
        labels_all.append(y.numpy())
        ratios_all.append(mask.mean(dim=1).cpu().numpy())

preds  = np.concatenate(preds_all)
labels = np.concatenate(labels_all)
ratios = np.concatenate(ratios_all)

buckets = np.arange(0, 1.01, 0.2)
miou_by_bucket = []
for lo, hi in zip(buckets[:-1], buckets[1:]):
    mask_b = (ratios >= lo) & (ratios < hi + 1e-9)
    if mask_b.sum() == 0:
        miou_by_bucket.append(float("nan"))
        continue
    p_b = preds[mask_b]; l_b = labels[mask_b]
    cm_b = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    idx_b = NUM_CLASSES * l_b + p_b
    cm_b += np.bincount(idx_b, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    m, _, _ = metrics_from_cm(cm_b)
    miou_by_bucket.append(m)
    print(f"valid_ratio [{lo:.0%}–{hi:.0%}): n={mask_b.sum():,}  mIoU={m:.4f}")

labels_b = [f"{int(lo*100)}–{int(hi*100)}%" for lo, hi in zip(buckets[:-1], buckets[1:])]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels_b, miou_by_bucket, color="#4C72B0")
ax.set_xlabel("Fraction of valid steps in window"); ax.set_ylabel("mIoU")
ax.set_title("mIoU by sparsity bucket (test set)")
ax.grid(axis="y", alpha=0.3)
for i, v in enumerate(miou_by_bucket):
    if not np.isnan(v):
        ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(RUN_DIR / "miou_by_sparsity.png", dpi=160)
plt.show()


## 14. Done

Files in `runs/lstm_sparse_aware_v1/`:

| file | what it is |
|---|---|
| `best.pt` | best-by-val checkpoint (mean/std included) |
| `metrics.csv` | per-epoch train/val curves + LR |
| `test_metrics.json` | test mIoU, pix_acc, per-class P/R/F1/IoU, macro-F1 |
| `confmat.png` | prof-style percentage confusion matrix |
| `loss_curve.png` | loss + mIoU + LR training diagnostics |
| `miou_by_sparsity.png` | mIoU per valid-ratio bucket (sparsity analysis) |

**What makes this different from `final.ipynb`:**

| | final.ipynb | lstm_sparse_aware.ipynb |
|---|---|---|
| Invalid steps | not handled | **zeroed out** |
| Pooling | mean over all steps | **masked mean over valid steps only** |
| valid_ratio feature | no | **yes (+1 dim to head input)** |
| Sparsity analysis | no | **yes (bucket plot)** |

If mIoU on the 0–20 % bucket (most-sparse windows) is meaningfully
higher than `final.ipynb`, the sparse-aware design is paying off.
